kaggle環境で実行したものです。

In [ ]:
import subprocess
import sys

print("="*60)
print("環境確認")
print("="*60)

# GPU情報
print("\n【GPU情報】")
!nvidia-smi

# Pythonバージョン
print("\n【Pythonバージョン】")
!python --version

# ディスク容量
print("\n【ディスク容量】")
!df -h /kaggle/working

In [ ]:
import sys

print("依存ライブラリをインストール中...")

# 基本的なライブラリ
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# LeRobotのインストール（最新安定版）
!pip install -q "lerobot>=0.3.0"

# 追加ライブラリ
!pip install -q pandas pyarrow pillow matplotlib opencv-python

print("✓ インストール完了")

# バージョン確認
import torch
import lerobot
print(f"\nPyTorch version: {torch.__version__}")
print(f"LeRobot version: {lerobot.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
import os
from pathlib import Path

print("="*60)
print("データセットの確認")
print("="*60)

# Kaggleにアップロードしたデータセットのパス
# 左側のデータパネルからデータセットを追加し、パスを確認してください
dataset_path = "/kaggle/input/datasets/naruhisahayashi/smolvla-dataset-mtl-3/dataset"  

print(f"\nデータセットパス: {dataset_path}")

# ディレクトリ構造の確認
if os.path.exists(dataset_path):
    print("\n【ディレクトリ構造】")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # 最初の5ファイルのみ表示
            print(f'{subindent}{file}')
        if len(files) > 5:
            print(f'{subindent}... and {len(files)-5} more files')
else:
    print(f"⚠️ 警告: {dataset_path} が見つかりません")
    print("左側のデータパネル（+ Add data）からデータセットを追加してください")

In [ ]:
!pip install 'lerobot[dataset]'

In [ ]:
import sys
import importlib.util
import importlib.metadata

print("Python:", sys.executable)
print("av spec:", importlib.util.find_spec("av"))

try:
    import av
    print("av version:", av.__version__)
    print("av location:", av.__file__)
except Exception as e:
    print("av import error:", repr(e))

try:
    print("Installed distribution:", importlib.metadata.version("av"))
except Exception as e:
    print("Distribution error:", repr(e))

In [ ]:
import lerobot.utils.import_utils as iu

print("Before:", iu._require_package_cache)

iu._require_package_cache.clear()

from lerobot.datasets.lerobot_dataset import LeRobotDataset

print("LeRobotDataset のインポート成功！")

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import json

print("="*60)
print("データセットの基本情報")
print("="*60)

try:
    # ローカルパスからデータセットをロード
    dataset = LeRobotDataset(dataset_path)
    
    print(f"\n✓ データセットの読み込みに成功")
    print(f"総エピソード数: {dataset.num_episodes}")
    print(f"総フレーム数: {dataset.num_frames}")
    print(f"FPS: {dataset.fps}")
    
    # 特徴量の確認
    print(f"\n【特徴量（Features）】")
    for key, value in dataset.features.items():
        print(f"  {key}: {value}")
        
except Exception as e:
    print(f"エラー: {e}")
    print("\n代替方法: 直接parquetファイルを読み込む")
    import pandas as pd
    train_dir = Path(dataset_path) / "train"
    parquet_files = list(train_dir.glob("*.parquet"))
    print(f"Parquetファイル数: {len(parquet_files)}")
    if parquet_files:
        df = pd.read_parquet(parquet_files[0])
        print(f"\n最初のエピソードのフレーム数: {len(df)}")
        print(f"カラム: {list(df.columns)}")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("="*60)
print("データの品質検査")
print("="*60)

train_dir = Path(dataset_path) / "train"
parquet_files = sorted(train_dir.glob("*.parquet"))

print(f"\n検査対象エピソード数: {len(parquet_files)}")

# 統計情報の集計
total_frames = 0
state_dims = []
action_dims = []
tasks = set()
missing_data = {
    "state": 0,
    "action": 0,
    "task": 0,
    "image": 0
}

for i, ep_file in enumerate(parquet_files):
    df = pd.read_parquet(ep_file)
    total_frames += len(df)
    
    # 次元の確認
    if i == 0:
        state_dims.append(len(df["observation.state"].iloc[0]))
        action_dims.append(len(df["action"].iloc[0]))
    
    # 欠損値のチェック
    missing_data["state"] += df["observation.state"].isnull().sum()
    missing_data["action"] += df["action"].isnull().sum()
    missing_data["task"] += df["task"].isnull().sum()
    
    # タスクの収集
    tasks.update(df["task"].unique())

print(f"\n【基本統計】")
print(f"  総フレーム数: {total_frames}")
print(f"  1エピソード平均フレーム数: {total_frames / len(parquet_files):.1f}")
print(f"  State次元: {state_dims[0]}")
print(f"  Action次元: {action_dims[0]}")
print(f"  一意のタスク数: {len(tasks)}")

print(f"\n【欠損値チェック】")
for key, count in missing_data.items():
    status = "✓" if count == 0 else "⚠️"
    print(f"  {status} {key}: {count}個")

if len(tasks) > 0:
    print(f"\n【タスク指示の例（最初の5個）】")
    for i, task in enumerate(list(tasks)[:5], 1):
        print(f"  {i}. {task}")

In [ ]:
import pandas as pd
from pathlib import Path
import json

print("="*60)
print("エピソード統計")
print("="*60)

# tasks.jsonlの読み込み
tasks_file = Path(dataset_path) / "meta" / "tasks.jsonl"
if tasks_file.exists():
    tasks_data = []
    with open(tasks_file, 'r') as f:
        for line in f:
            tasks_data.append(json.loads(line))
    
    print(f"\n【タスク分布】")
    print(f"  総タスク数: {len(tasks_data)}")
    
    # info.jsonの読み込み
    info_file = Path(dataset_path) / "meta" / "info.json"
    if info_file.exists():
        with open(info_file, 'r') as f:
            info = json.load(f)
        
        print(f"\n【データセット情報】")
        print(f"  Codebase version: {info.get('codebase_version', 'N/A')}")
        print(f"  Robot type: {info.get('robot_type', 'N/A')}")
        print(f"  FPS: {info.get('fps', 'N/A')}")
        print(f"  Total episodes: {info.get('total_episodes', 'N/A')}")

# 各エピソードのフレーム数
train_dir = Path(dataset_path) / "train"
parquet_files = sorted(train_dir.glob("*.parquet"))

frame_counts = []
for ep_file in parquet_files:
    df = pd.read_parquet(ep_file)
    frame_counts.append(len(df))

print(f"\n【エピソード長さの統計】")
print(f"  最小: {min(frame_counts)} フレーム")
print(f"  最大: {max(frame_counts)} フレーム")
print(f"  平均: {sum(frame_counts)/len(frame_counts):.1f} フレーム")
print(f"  中央値: {sorted(frame_counts)[len(frame_counts)//2]} フレーム")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

print("="*60)
print("サンプルデータの可視化")
print("="*60)

# 最初のエピソードからサンプルを取得
train_dir = Path(dataset_path) / "train"
parquet_files = sorted(train_dir.glob("*.parquet"))

if parquet_files:
    df = pd.read_parquet(parquet_files[0])
    
    # ランダムに3フレーム選択
    sample_indices = np.random.choice(len(df), min(3, len(df)), replace=False)
    
    fig, axes = plt.subplots(1, len(sample_indices), figsize=(15, 5))
    if len(sample_indices) == 1:
        axes = [axes]
    
    for idx, ax in zip(sample_indices, axes):
        row = df.iloc[idx]
        
        # 画像の読み込み
        ep_num = int(row['episode_index'])
        frame_num = int(row['frame_index'])
        img_path = Path(dataset_path) / "train" / "images" / "observation.images.cam_overhead" / f"episode_{ep_num:06d}" / f"frame_{frame_num:06d}.jpg"
        
        if img_path.exists():
            from PIL import Image
            img = Image.open(img_path)
            ax.imshow(img)
            ax.axis('off')
            ax.set_title(f"Frame {frame_num}\n{row['task'][:40]}...", fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    # StateとActionの可視化
    print("\n【StateとActionの統計（最初のエピソード）】")
    print(f"  State range: [{df['observation.state'].apply(lambda x: np.array(x).min()).min():.3f}, {df['observation.state'].apply(lambda x: np.array(x).max()).max():.3f}]")
    print(f"  Action range: [{df['action'].apply(lambda x: np.array(x).min()).min():.3f}, {df['action'].apply(lambda x: np.array(x).max()).max():.3f}]")
else:
    print("⚠️ Parquetファイルが見つかりません")

In [ ]:
import os

for current, dirs, files in os.walk(dataset_path):
    level = current.replace(dataset_path, "").count(os.sep)

    if level > 2:
        dirs[:] = []
        continue

    print("  " * level + os.path.basename(current) + "/")

    for filename in files[:8]:
        print("  " * (level + 1) + filename)

In [ ]:
import json
import os

dataset_path = (
    "/kaggle/input/datasets/naruhisahayashi/smolvla-dataset-mtl-3/dataset"
)

info_path = os.path.join(dataset_path, "meta", "info.json")

with open(info_path, "r") as f:
    info = json.load(f)

print(json.dumps(info, indent=2, ensure_ascii=False))

In [ ]:
import glob
import pyarrow.parquet as pq

parquet_files = glob.glob(
    os.path.join(dataset_path, "train", "*.parquet")
)

print("Parquet ファイル数:", len(parquet_files))

if parquet_files:
    table = pq.read_table(parquet_files[0])

    print("\n列名:")
    print(table.column_names)

    print("\nスキーマ:")
    print(table.schema)

    print("\n先頭2行:")
    print(table.slice(0, 2).to_pylist())

In [ ]:
import os
import glob
import json
import pyarrow.parquet as pq

dataset_path = (
    "/kaggle/input/datasets/naruhisahayashi/smolvla-dataset-mtl-3/dataset"
)

# info.json の全内容
with open(os.path.join(dataset_path, "meta", "info.json")) as f:
    info = json.load(f)

print("=== info.json のキー ===")
print(list(info.keys()))

print("\n=== Parquet の列とスキーマ ===")
files = sorted(glob.glob(os.path.join(dataset_path, "train", "*.parquet")))
print("ファイル数:", len(files))

table = pq.read_table(files[0])
print(table.schema)
print("\n列名:", table.column_names)

print("\n=== 最初の行（画像データは省略） ===")
row = table.slice(0, 1).to_pylist()[0]

for key, value in row.items():
    if isinstance(value, dict):
        print(key, "->", {
            k: (f"<データ省略: {len(v)} bytes>" if isinstance(v, bytes) else type(v).__name__)
            for k, v in value.items()
        })
    elif isinstance(value, bytes):
        print(key, f"<データ省略: {len(value)} bytes>")
    else:
        print(key, repr(value)[:300])

In [ ]:
tasks_path = os.path.join(dataset_path, "meta", "tasks.jsonl")

with open(tasks_path, encoding="utf-8") as f:
    for line in list(f)[:5]:
        print(line.rstrip())

In [ ]:
import os
from collections import Counter

images_dir = os.path.join(dataset_path, "train", "images")

print("画像フォルダの存在:", os.path.isdir(images_dir))

if os.path.isdir(images_dir):
    all_files = []
    for root, dirs, files in os.walk(images_dir):
        all_files.extend(
            os.path.join(root, f)
            for f in files
        )

    print("画像フォルダ内のファイル数:", len(all_files))
    print("拡張子ごとの件数:",
          Counter(os.path.splitext(f)[1].lower() for f in all_files))

    print("\n先頭20ファイル:")
    for f in sorted(all_files)[:20]:
        print(os.path.relpath(f, images_dir))

In [ ]:

import os
import glob
import json
import shutil
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from itertools import zip_longest
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# ============================================================
# 1. 入出力パス
# ============================================================
src = Path(
    "/kaggle/input/datasets/naruhisahayashi/smolvla-dataset-mtl-3/dataset"
)

out = Path(
    "/kaggle/working/train-smolvla-mtl-dataset-lerobot"
)

assert src.is_dir(), f"入力フォルダがありません: {src}"
assert (src / "meta" / "info.json").is_file(), "info.json がありません"

# 既存の出力を削除
if out.exists():
    print("出力先を削除します。")
    shutil.rmtree(out)

# ============================================================
# 2. 元データの読み込み・検証・修正
# ============================================================
with open(src / "meta" / "info.json", encoding="utf-8") as f:
    source_info = json.load(f)

parquet_files = sorted((src / "train").glob("episode_*.parquet"))

episodes = []
image_root = src / "train" / "images"

# 削除するエピソード
DELETE_EPISODES = {17, 23, 26, 27, 54}

# 赤・青・緑のタスク名を判定する
def get_task_color(task):
    task_lower = task.lower()

    if "red" in task_lower:
        return "red"
    elif "blue" in task_lower:
        return "blue"
    elif "green" in task_lower:
        return "green"
    else:
        raise ValueError(f"色を判定できないタスクです: {task}")


# 赤と青のタスクラベルを入れ替える
def swap_red_blue(task):
    if "red" in task.lower():
        return task.replace("red", "TEMP_COLOR").replace(
            "blue", "red"
        ).replace("TEMP_COLOR", "blue")

    elif "blue" in task.lower():
        return task.replace("blue", "TEMP_COLOR").replace(
            "red", "blue"
        ).replace("TEMP_COLOR", "red")

    return task


for p in parquet_files:
    df = pd.read_parquet(p)
    df = df.sort_values("frame_index").reset_index(drop=True)

    assert df["episode_index"].nunique() == 1, (
        f"{p.name}: 複数の episode_index があります"
    )

    ep_idx = int(df["episode_index"].iloc[0])

    # --------------------------------------------------------
    # 指定エピソードを除外
    # --------------------------------------------------------
    if ep_idx in DELETE_EPISODES:
        print(f"Episode {ep_idx}: 除外")
        continue

    # --------------------------------------------------------
    # frame_index の検証
    # --------------------------------------------------------
    assert df["frame_index"].tolist() == list(range(len(df))), (
        f"Episode {ep_idx}: frame_index に欠番があります"
    )

    tasks = df["task"].dropna().unique().tolist()

    assert len(tasks) == 1, (
        f"Episode {ep_idx}: task が1種類ではありません: {tasks}"
    )

    original_task = tasks[0]

    # --------------------------------------------------------
    # 赤・青のタスクラベルを入れ替え
    # --------------------------------------------------------
    corrected_task = swap_red_blue(original_task)

    if corrected_task != original_task:
        print(
            f"Episode {ep_idx}: "
            f"{original_task} -> {corrected_task}"
        )

    # DataFrame 内の task も修正
    df["task"] = corrected_task

    # 修正後の色を判定
    color = get_task_color(corrected_task)

    # --------------------------------------------------------
    # 画像の存在確認
    # --------------------------------------------------------
    ep_image_dir = (
        image_root
        / "observation.images.camera1"
        / f"episode_{ep_idx:06d}"
    )

    for frame_idx in df["frame_index"]:
        img_path = ep_image_dir / f"frame_{int(frame_idx):06d}.jpg"
        assert img_path.is_file(), f"画像がありません: {img_path}"

    episodes.append({
        "original_ep_idx": ep_idx,
        "df": df,
        "task": corrected_task,
        "color": color,
    })


# ============================================================
# 3. 赤 → 青 → 緑 の順番に並べ替え
# ============================================================

red_episodes = sorted(
    [ep for ep in episodes if ep["color"] == "red"],
    key=lambda ep: ep["original_ep_idx"]
)

blue_episodes = sorted(
    [ep for ep in episodes if ep["color"] == "blue"],
    key=lambda ep: ep["original_ep_idx"]
)

green_episodes = sorted(
    [ep for ep in episodes if ep["color"] == "green"],
    key=lambda ep: ep["original_ep_idx"]
)

print("\n色ごとのエピソード数")
print("Red:", len(red_episodes))
print("Blue:", len(blue_episodes))
print("Green:", len(green_episodes))

# 3色のエピソード数が同じであることを確認
# assert (
#     len(red_episodes)
#     == len(blue_episodes)
#     == len(green_episodes)
# ), (
#     "赤・青・緑のエピソード数が一致しません。"
#     "順番を確認してください。"
# )

# 赤 → 青 → 緑 の順番で交互に並べる
ordered_episodes = []

for red_ep, blue_ep, green_ep in zip(
    red_episodes,
    blue_episodes,
    green_episodes
):
    ordered_episodes.extend([
        red_ep,
        blue_ep,
        green_ep,
    ])

episodes = ordered_episodes

print("\n並べ替え後の順番:")
for new_idx, ep in enumerate(episodes):
    print(
        f"{new_idx:03d}: "
        f"元Episode {ep['original_ep_idx']:03d} "
        f"({ep['color']})"
    )

print("\nエピソード数:", len(episodes))
print(
    "総フレーム数:",
    sum(len(ep["df"]) for ep in episodes)
)


# ============================================================
# 4. LeRobot データセットを新規作成
# ============================================================
features = {
    "observation.images.camera1": {
        "dtype": "video",
        "shape": (3, 480, 640),
        "names": ["channels", "height", "width"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (6,),
        "names": [f"joint_{i}" for i in range(6)],
    },
    "action": {
        "dtype": "float32",
        "shape": (6,),
        "names": [f"joint_{i}" for i in range(6)],
    },
}

dataset = LeRobotDataset.create(
    repo_id="local/train-smolvla-mtl-dataset-lerobot",
    root=out,
    fps=int(source_info["fps"]),
    robot_type=source_info["robot_type"],
    features=features,
    use_videos=True,
)


# ============================================================
# 5. 各エピソードを読み込み、登録
# ============================================================
for n, ep in enumerate(episodes, start=1):

    ep_idx = ep["original_ep_idx"]
    df = ep["df"]
    task = ep["task"]

    print(
        f"\r変換中: 元Episode {ep_idx} "
        f"({n}/{len(episodes)}) [{ep['color']}]",
        end="",
        flush=True,
    )

    for _, row in df.iterrows():

        frame_idx = int(row["frame_index"])

        image_path = (
            src
            / "train"
            / "images"
            / "observation.images.camera1"
            / f"episode_{ep_idx:06d}"
            / f"frame_{frame_idx:06d}.jpg"
        )

        with Image.open(image_path) as im:
            image = np.array(im.convert("RGB"))

        frame = {
            "observation.images.camera1": image,

            "observation.state": np.asarray(
                row["observation.state"],
                dtype=np.float32
            ),

            "action": np.asarray(
                row["action"],
                dtype=np.float32
            ),

            "task": task,
        }

        dataset.add_frame(frame)

    dataset.save_episode()


# ============================================================
# 6. 書き込み完了
# ============================================================
dataset.finalize()

print("\n\n変換完了:", out)
print("最終エピソード数:", len(episodes))
